In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/race_rc_project"

PROCESSED_DIR = f"{PROJECT}/data/processed"
MODEL_B_DIR = f"{PROJECT}/models/model_b"
RESULTS_DIR = f"{PROJECT}/results"
MODEL_B_RESULTS_DIR = f"{RESULTS_DIR}/model_b_true_full"

TRAIN_PATH = f"{PROCESSED_DIR}/model_b_train_full.csv"
VAL_PATH = f"{PROCESSED_DIR}/model_b_val_full.csv"
TEST_PATH = f"{PROCESSED_DIR}/model_b_test_full.csv"

os.makedirs(MODEL_B_DIR, exist_ok=True)
os.makedirs(MODEL_B_RESULTS_DIR, exist_ok=True)

print("PROJECT:", PROJECT)
print("PROJECT exists:", os.path.exists(PROJECT))
print("MODEL_B_RESULTS_DIR exists:", os.path.exists(MODEL_B_RESULTS_DIR))

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    print(p, "->", os.path.exists(p))

Mounted at /content/drive
PROJECT: /content/drive/MyDrive/race_rc_project
PROJECT exists: True
MODEL_B_RESULTS_DIR exists: True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_train_full.csv -> True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_val_full.csv -> True
/content/drive/MyDrive/race_rc_project/data/processed/model_b_test_full.csv -> True


In [2]:
!pip -q install rouge-score nltk

  Preparing metadata (setup.py) ... done


In [3]:
import re
import ast
import json
import math
import string
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import joblib

import nltk
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

FULL_MODE = True
MAX_NEGATIVE_CANDIDATES_PER_QUESTION = 40
FEATURE_CHUNK_SIZE = 100000
SCORE_CHUNK_SIZE = 100000

print("Settings loaded.")

Settings loaded.


In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

Train shape: (70280, 16)
Val shape: (8785, 16)
Test shape: (8786, 16)

Columns:
['question_uid', 'id', 'article', 'article_nopunct', 'question', 'question_nopunct', 'correct_answer', 'correct_answer_nopunct', 'gold_answer_label', 'distractor_1', 'distractor_2', 'distractor_3', 'distractor_label_1', 'distractor_label_2', 'distractor_label_3', 'article_sentences']


In [5]:
tfidf_path = f"{MODEL_B_DIR}/model_b_tfidf_vectorizer_full.pkl"

if not os.path.exists(tfidf_path):
    raise FileNotFoundError(f"Missing TF-IDF vectorizer: {tfidf_path}")

model_b_tfidf = joblib.load(tfidf_path)

print("Loaded TF-IDF vectorizer:", tfidf_path)
print("Vocabulary size:", len(model_b_tfidf.vocabulary_))

Loaded TF-IDF vectorizer: /content/drive/MyDrive/race_rc_project/models/model_b/model_b_tfidf_vectorizer_full.pkl
Vocabulary size: 50000


In [6]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x)

def normalize_text(text):
    text = safe_text(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_punctuation(text):
    return safe_text(text).translate(str.maketrans("", "", string.punctuation))

def tokenize_words(text):
    text = normalize_text(remove_punctuation(text))
    return re.findall(r"\b[a-zA-Z][a-zA-Z0-9]*\b", text)

def content_words(text):
    return [t for t in tokenize_words(text) if t not in STOPWORDS and len(t) > 2]

def word_overlap(a, b):
    a_words = set(content_words(a))
    b_words = set(content_words(b))
    if not a_words or not b_words:
        return 0.0
    return len(a_words & b_words) / len(a_words | b_words)

def char_overlap(a, b):
    a_chars = set(normalize_text(remove_punctuation(a)).replace(" ", ""))
    b_chars = set(normalize_text(remove_punctuation(b)).replace(" ", ""))
    if not a_chars or not b_chars:
        return 0.0
    return len(a_chars & b_chars) / len(a_chars | b_chars)

def candidate_frequency_in_article(candidate, article):
    cand = normalize_text(candidate)
    art = normalize_text(article)
    if not cand:
        return 0
    return art.count(cand)

def count_csv_rows(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f) - 1

print("Text helpers ready.")

Text helpers ready.


In [7]:
def generate_ngrams(tokens, n):
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def extract_candidate_phrases(article, question=None, correct_answer=None, max_candidates=40):
    article_norm = normalize_text(article)
    question_norm = normalize_text(question)
    answer_norm = normalize_text(correct_answer)

    tokens = content_words(article_norm)

    unigrams = tokens
    bigrams = generate_ngrams(tokens, 2)
    trigrams = generate_ngrams(tokens, 3)

    all_candidates = unigrams + bigrams + trigrams
    counts = Counter(all_candidates)

    cleaned = []
    seen = set()

    for cand, freq in counts.most_common():
        cand_norm = normalize_text(cand)

        if cand_norm in seen:
            continue
        if len(cand_norm) < 3:
            continue
        if cand_norm == answer_norm:
            continue
        if answer_norm and cand_norm in answer_norm:
            continue
        if answer_norm and answer_norm in cand_norm:
            continue
        if answer_norm and word_overlap(cand_norm, answer_norm) >= 0.85:
            continue
        if question_norm and cand_norm == question_norm:
            continue

        cleaned.append(cand_norm)
        seen.add(cand_norm)

        if len(cleaned) >= max_candidates:
            break

    return cleaned

def get_gold_distractors(row):
    distractors = [
        safe_text(row["distractor_1"]).strip(),
        safe_text(row["distractor_2"]).strip(),
        safe_text(row["distractor_3"]).strip()
    ]
    return [d for d in distractors if d]

def build_candidate_rows_for_question(row, max_negative_candidates=40):
    question_uid = row["question_uid"]
    article = safe_text(row["article"])
    question = safe_text(row["question"])
    correct_answer = safe_text(row["correct_answer"])

    gold_distractors = get_gold_distractors(row)
    gold_norm = set(normalize_text(d) for d in gold_distractors)

    rows = []

    for d in gold_distractors:
        rows.append({
            "question_uid": question_uid,
            "article": article,
            "question": question,
            "correct_answer": correct_answer,
            "candidate": d,
            "label": 1,
            "source": "gold_distractor"
        })

    extracted = extract_candidate_phrases(
        article=article,
        question=question,
        correct_answer=correct_answer,
        max_candidates=max_negative_candidates
    )

    for cand in extracted:
        cand_norm = normalize_text(cand)

        if cand_norm in gold_norm:
            continue

        rows.append({
            "question_uid": question_uid,
            "article": article,
            "question": question,
            "correct_answer": correct_answer,
            "candidate": cand,
            "label": 0,
            "source": "extracted_negative"
        })

    return rows

print("Candidate functions ready.")

Candidate functions ready.


In [8]:
feature_cols = [
    "cos_candidate_answer",
    "cos_candidate_question",
    "cos_candidate_article",
    "candidate_word_len",
    "answer_word_len",
    "length_diff",
    "char_overlap_answer",
    "word_overlap_answer",
    "word_overlap_question",
    "candidate_frequency_article",
    "candidate_in_question",
    "candidate_same_as_answer",
    "candidate_contains_answer",
    "answer_contains_candidate"
]

def paired_cosine_sparse(A, B):
    numer = A.multiply(B).sum(axis=1).A1
    A_norm = np.sqrt(A.multiply(A).sum(axis=1).A1)
    B_norm = np.sqrt(B.multiply(B).sum(axis=1).A1)

    denom = A_norm * B_norm
    out = np.zeros_like(numer, dtype=float)

    mask = denom != 0
    out[mask] = numer[mask] / denom[mask]

    return out

def extract_features_chunk(chunk_df):
    chunk_df = chunk_df.copy()

    candidates = chunk_df["candidate"].fillna("").astype(str).tolist()
    answers = chunk_df["correct_answer"].fillna("").astype(str).tolist()
    questions = chunk_df["question"].fillna("").astype(str).tolist()
    articles = chunk_df["article"].fillna("").astype(str).tolist()

    cand_vec = model_b_tfidf.transform(candidates)
    ans_vec = model_b_tfidf.transform(answers)
    ques_vec = model_b_tfidf.transform(questions)
    art_vec = model_b_tfidf.transform(articles)

    cos_candidate_answer = paired_cosine_sparse(cand_vec, ans_vec)
    cos_candidate_question = paired_cosine_sparse(cand_vec, ques_vec)
    cos_candidate_article = paired_cosine_sparse(cand_vec, art_vec)

    rows = []

    for idx, row in chunk_df.reset_index(drop=True).iterrows():
        article = safe_text(row["article"])
        question = safe_text(row["question"])
        correct_answer = safe_text(row["correct_answer"])
        candidate = safe_text(row["candidate"])

        cand_words = content_words(candidate)
        ans_words = content_words(correct_answer)

        rows.append({
            "label": int(row["label"]),

            "cos_candidate_answer": float(cos_candidate_answer[idx]),
            "cos_candidate_question": float(cos_candidate_question[idx]),
            "cos_candidate_article": float(cos_candidate_article[idx]),

            "candidate_word_len": len(cand_words),
            "answer_word_len": len(ans_words),
            "length_diff": abs(len(cand_words) - len(ans_words)),

            "char_overlap_answer": char_overlap(candidate, correct_answer),
            "word_overlap_answer": word_overlap(candidate, correct_answer),
            "word_overlap_question": word_overlap(candidate, question),

            "candidate_frequency_article": candidate_frequency_in_article(candidate, article),
            "candidate_in_question": int(normalize_text(candidate) in normalize_text(question)),
            "candidate_same_as_answer": int(normalize_text(candidate) == normalize_text(correct_answer)),
            "candidate_contains_answer": int(normalize_text(correct_answer) in normalize_text(candidate)),
            "answer_contains_candidate": int(normalize_text(candidate) in normalize_text(correct_answer))
        })

    return pd.DataFrame(rows)

print("Feature extraction functions ready.")

Feature extraction functions ready.


In [9]:
train_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_train_candidates_full.csv"

def build_candidate_dataset_train_safe(df, output_path, max_negative_candidates=40):
    temp_path = output_path + ".tmp"

    if os.path.exists(output_path):
        print("Train candidate file already exists:", output_path)
        return pd.read_csv(output_path)

    if os.path.exists(temp_path):
        os.remove(temp_path)
        print("Removed old temp candidate file:", temp_path)

    first_write = True
    total_written = 0
    total_positive = 0
    total_negative = 0

    for i, row in df.iterrows():
        rows = build_candidate_rows_for_question(
            row,
            max_negative_candidates=max_negative_candidates
        )

        chunk_df = pd.DataFrame(rows)

        if len(chunk_df) > 0:
            chunk_df.to_csv(
                temp_path,
                mode="w" if first_write else "a",
                index=False,
                header=first_write
            )

            first_write = False
            total_written += len(chunk_df)
            total_positive += int((chunk_df["label"] == 1).sum())
            total_negative += int((chunk_df["label"] == 0).sum())

        if (i + 1) % 5000 == 0:
            print(
                f"Processed {i + 1}/{len(df)} questions | "
                f"rows={total_written}, pos={total_positive}, neg={total_negative}"
            )

    os.rename(temp_path, output_path)

    print("\nSaved train candidate file:", output_path)
    print("Total rows:", total_written)
    print("Positive labels:", total_positive)
    print("Negative labels:", total_negative)

    return pd.read_csv(output_path)

train_candidates = build_candidate_dataset_train_safe(
    train_df,
    train_candidates_path,
    max_negative_candidates=MAX_NEGATIVE_CANDIDATES_PER_QUESTION
)

print("\ntrain_candidates shape:", train_candidates.shape)
print(train_candidates["label"].value_counts())
print("Unique train questions:", train_candidates["question_uid"].nunique())

expected_pos = len(train_df) * 3
actual_pos = int((train_candidates["label"] == 1).sum())

print("Expected positives:", expected_pos)
print("Actual positives:", actual_pos)

if actual_pos != expected_pos:
    raise ValueError("Train candidate positive count is wrong.")

if train_candidates["question_uid"].nunique() != len(train_df):
    raise ValueError("Not all train questions are represented.")

print("Train candidate file is correct.")

Processed 5000/70280 questions | rows=214696, pos=15000, neg=199696
Processed 10000/70280 questions | rows=429447, pos=30000, neg=399447
Processed 15000/70280 questions | rows=644186, pos=45000, neg=599186
Processed 20000/70280 questions | rows=858890, pos=60000, neg=798890
Processed 25000/70280 questions | rows=1073559, pos=75000, neg=998559
Processed 30000/70280 questions | rows=1288224, pos=90000, neg=1198224
Processed 35000/70280 questions | rows=1502968, pos=105000, neg=1397968
Processed 40000/70280 questions | rows=1717669, pos=120000, neg=1597669
Processed 45000/70280 questions | rows=1932410, pos=135000, neg=1797410
Processed 50000/70280 questions | rows=2147099, pos=150000, neg=1997099
Processed 55000/70280 questions | rows=2361865, pos=165000, neg=2196865
Processed 60000/70280 questions | rows=2576613, pos=180000, neg=2396613
Processed 65000/70280 questions | rows=2791321, pos=195000, neg=2596321
Processed 70000/70280 questions | rows=3005999, pos=210000, neg=2795999

Saved t

In [10]:
train_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_train_features_full.csv"

def extract_train_features_safe_resumable(candidate_path, output_path, chunk_size=100000):
    temp_path = output_path + ".tmp"

    total_rows = count_csv_rows(candidate_path)
    final_rows = count_csv_rows(output_path)
    temp_rows = count_csv_rows(temp_path)

    print("Candidate rows:", total_rows)
    print("Existing final feature rows:", final_rows)
    print("Existing temp feature rows:", temp_rows)

    if os.path.exists(output_path) and final_rows == total_rows:
        print("Complete train feature file already exists.")
        return pd.read_csv(output_path)

    if os.path.exists(output_path) and final_rows != total_rows:
        print("Incomplete/corrupt final train feature file found. Deleting it.")
        os.remove(output_path)

    resume_from = 0
    first_write = True
    mode = "w"

    if os.path.exists(temp_path) and temp_rows > 0:
        if temp_rows % chunk_size == 0:
            resume_from = temp_rows
            first_write = False
            mode = "a"
            print(f"Resuming from existing temp file at row {resume_from}.")
        else:
            print("Temp file row count is not chunk-aligned. Rebuilding from scratch.")
            os.remove(temp_path)
            resume_from = 0
            first_write = True
            mode = "w"

    processed = 0

    for chunk in pd.read_csv(candidate_path, chunksize=chunk_size):
        chunk_start = processed
        chunk_end = processed + len(chunk)

        if chunk_end <= resume_from:
            processed = chunk_end
            continue

        if chunk_start < resume_from < chunk_end:
            raise ValueError(
                "Resume point falls inside a chunk. Delete temp file and rebuild from scratch."
            )

        feature_chunk = extract_features_chunk(chunk)

        keep_cols = ["label"] + feature_cols
        feature_chunk = feature_chunk[keep_cols].copy()
        feature_chunk["label"] = feature_chunk["label"].astype(int)

        feature_chunk.to_csv(
            temp_path,
            mode=mode,
            index=False,
            header=first_write
        )

        first_write = False
        mode = "a"

        processed = chunk_end
        print(f"Processed train feature rows: {processed}/{total_rows}")

    temp_rows_after = count_csv_rows(temp_path)

    print("Temp rows after extraction:", temp_rows_after)

    if temp_rows_after != total_rows:
        raise ValueError(
            f"Train feature row mismatch. Expected {total_rows}, got {temp_rows_after}."
        )

    os.rename(temp_path, output_path)

    print("\nSaved complete train features:", output_path)

    return pd.read_csv(output_path)

train_features = extract_train_features_safe_resumable(
    train_candidates_path,
    train_features_path,
    chunk_size=FEATURE_CHUNK_SIZE
)

Candidate rows: 3018028
Existing final feature rows: 0
Existing temp feature rows: 100000
Resuming from existing temp file at row 100000.
Processed train feature rows: 200000/3018028
Processed train feature rows: 300000/3018028
Processed train feature rows: 400000/3018028
Processed train feature rows: 500000/3018028
Processed train feature rows: 600000/3018028
Processed train feature rows: 700000/3018028
Processed train feature rows: 800000/3018028
Processed train feature rows: 900000/3018028
Processed train feature rows: 1000000/3018028
Processed train feature rows: 1100000/3018028
Processed train feature rows: 1200000/3018028
Processed train feature rows: 1300000/3018028
Processed train feature rows: 1400000/3018028
Processed train feature rows: 1500000/3018028
Processed train feature rows: 1600000/3018028
Processed train feature rows: 1700000/3018028
Processed train feature rows: 1800000/3018028
Processed train feature rows: 1900000/3018028
Processed train feature rows: 2000000/3018

In [11]:
train_features = pd.read_csv(train_features_path)

print("train_features shape:", train_features.shape)
print(train_features["label"].value_counts())

bad_labels = set(train_features["label"].unique()) - {0, 1}
print("Bad labels:", bad_labels)

expected_rows = len(train_candidates)
actual_rows = len(train_features)

print("Expected feature rows:", expected_rows)
print("Actual feature rows:", actual_rows)

if actual_rows != expected_rows:
    raise ValueError("Train feature row count does not match train candidate row count.")

if bad_labels:
    raise ValueError("Train feature file has invalid labels.")

expected_cols = ["label"] + feature_cols
missing_cols = [c for c in expected_cols if c not in train_features.columns]
if missing_cols:
    raise ValueError(f"Missing train feature columns: {missing_cols}")

print("Train features are clean.")

train_features shape: (3018028, 15)
label
0    2807188
1     210840
Name: count, dtype: int64
Bad labels: set()
Expected feature rows: 3018028
Actual feature rows: 3018028
Train features are clean.


In [12]:
val_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_val_candidates_full.csv"
test_candidates_path = f"{MODEL_B_RESULTS_DIR}/model_b_test_candidates_full.csv"

val_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_val_features_full.csv"
test_features_path = f"{MODEL_B_RESULTS_DIR}/model_b_test_features_full.csv"

required_existing = [
    val_candidates_path,
    test_candidates_path,
    val_features_path,
    test_features_path
]

for p in required_existing:
    print(p, "->", os.path.exists(p))
    if not os.path.exists(p):
        raise FileNotFoundError(p)

val_candidates = pd.read_csv(val_candidates_path)
test_candidates = pd.read_csv(test_candidates_path)

val_features = pd.read_csv(val_features_path)
test_features = pd.read_csv(test_features_path)

print("\nval_candidates:", val_candidates.shape)
print(val_candidates["label"].value_counts())

print("\ntest_candidates:", test_candidates.shape)
print(test_candidates["label"].value_counts())

print("\nval_features:", val_features.shape)
print(val_features["label"].value_counts())

print("\ntest_features:", test_features.shape)
print(test_features["label"].value_counts())

/content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_val_candidates_full.csv -> True
/content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_test_candidates_full.csv -> True
/content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_val_features_full.csv -> True
/content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_test_features_full.csv -> True

val_candidates: (377205, 7)
label
0    350850
1     26355
Name: count, dtype: int64

test_candidates: (377353, 7)
label
0    350995
1     26358
Name: count, dtype: int64

val_features: (377205, 17)
label
0    350850
1     26355
Name: count, dtype: int64

test_features: (377353, 17)
label
0    350995
1     26358
Name: count, dtype: int64


In [13]:
X_train = train_features[feature_cols]
y_train = train_features["label"].astype(int)

X_val = val_features[feature_cols]
y_val = val_features["label"].astype(int)

X_test = test_features[feature_cols]
y_test = test_features["label"].astype(int)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("\nTrain labels:")
print(y_train.value_counts())

print("\nVal labels:")
print(y_val.value_counts())

print("\nTest labels:")
print(y_test.value_counts())

X_train: (3018028, 14)
X_val: (377205, 14)
X_test: (377353, 14)

Train labels:
label
0    2807188
1     210840
Name: count, dtype: int64

Val labels:
label
0    350850
1     26355
Name: count, dtype: int64

Test labels:
label
0    350995
1     26358
Name: count, dtype: int64


In [14]:
logreg_ranker_path = f"{MODEL_B_DIR}/logistic_regression_distractor_ranker_full.pkl"

if os.path.exists(logreg_ranker_path):
    print("Existing Logistic Regression model found. Deleting and retraining:", logreg_ranker_path)
    os.remove(logreg_ranker_path)

logreg_ranker = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

logreg_ranker.fit(X_train, y_train)
joblib.dump(logreg_ranker, logreg_ranker_path)

print("Saved Logistic Regression ranker:", logreg_ranker_path)

Saved Logistic Regression ranker: /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_distractor_ranker_full.pkl


In [15]:
rf_ranker_path = f"{MODEL_B_DIR}/random_forest_distractor_ranker_full.pkl"

if os.path.exists(rf_ranker_path):
    print("Existing Random Forest model found. Deleting and retraining:", rf_ranker_path)
    os.remove(rf_ranker_path)

rf_ranker = RandomForestClassifier(
    n_estimators=200,
    max_depth=18,
    min_samples_leaf=3,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

rf_ranker.fit(X_train, y_train)
joblib.dump(rf_ranker, rf_ranker_path)

print("Saved Random Forest ranker:", rf_ranker_path)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  3.8min
[Parallel(n_jobs=-1)]: Done 196 tasks      | elapsed: 15.0min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 15.3min finished


Saved Random Forest ranker: /content/drive/MyDrive/race_rc_project/models/model_b/random_forest_distractor_ranker_full.pkl


In [16]:
def score_feature_file(model, feature_path, candidate_path, output_path, model_name, chunk_size=100000):
    total_rows = count_csv_rows(feature_path)
    existing_rows = count_csv_rows(output_path)

    print(f"{model_name}: total feature rows:", total_rows)
    print(f"{model_name}: existing scored rows:", existing_rows)

    if os.path.exists(output_path) and existing_rows == total_rows:
        print(f"{model_name}: complete scored file already exists. Loading:", output_path)
        return pd.read_csv(output_path)

    if os.path.exists(output_path) and existing_rows != total_rows:
        print(f"{model_name}: incomplete scored file found. Deleting it.")
        os.remove(output_path)

    first_write = True
    processed = 0

    feature_chunks = pd.read_csv(feature_path, chunksize=chunk_size)
    candidate_chunks = pd.read_csv(candidate_path, chunksize=chunk_size)

    for f_chunk, c_chunk in zip(feature_chunks, candidate_chunks):
        X_chunk = f_chunk[feature_cols]

        if hasattr(model, "predict_proba"):
            scores = model.predict_proba(X_chunk)[:, 1]
        else:
            scores = model.decision_function(X_chunk)

        scored_chunk = c_chunk[[
            "question_uid",
            "question",
            "correct_answer",
            "candidate",
            "label",
            "source"
        ]].copy()

        scored_chunk["score"] = scores
        scored_chunk["model"] = model_name

        scored_chunk.to_csv(
            output_path,
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )

        first_write = False
        processed += len(scored_chunk)
        print(f"{model_name}: scored {processed}/{total_rows} rows")

    scored_rows = count_csv_rows(output_path)

    if scored_rows != total_rows:
        raise ValueError(
            f"{model_name}: scored row mismatch. Expected {total_rows}, got {scored_rows}."
        )

    scored_df = pd.read_csv(output_path)
    print(f"{model_name}: saved scored candidates:", output_path)
    print(scored_df.shape)

    return scored_df

print("Scoring helper ready.")

Scoring helper ready.


In [17]:
lr_val_scored_path = f"{MODEL_B_RESULTS_DIR}/val_scored_logreg_full.csv"
rf_val_scored_path = f"{MODEL_B_RESULTS_DIR}/val_scored_rf_full.csv"

for p in [lr_val_scored_path, rf_val_scored_path]:
    if os.path.exists(p):
        os.remove(p)
        print("Deleted old scored file:", p)

val_scored_lr = score_feature_file(
    logreg_ranker,
    val_features_path,
    val_candidates_path,
    lr_val_scored_path,
    "logistic_regression",
    SCORE_CHUNK_SIZE
)

val_scored_rf = score_feature_file(
    rf_ranker,
    val_features_path,
    val_candidates_path,
    rf_val_scored_path,
    "random_forest",
    SCORE_CHUNK_SIZE
)

logistic_regression: total feature rows: 377205
logistic_regression: existing scored rows: 0
logistic_regression: scored 100000/377205 rows
logistic_regression: scored 200000/377205 rows
logistic_regression: scored 300000/377205 rows
logistic_regression: scored 377205/377205 rows
logistic_regression: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/val_scored_logreg_full.csv
(377205, 8)
random_forest: total feature rows: 377205
random_forest: existing scored rows: 0


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.5s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.6s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.6s finished


random_forest: scored 100000/377205 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 200000/377205 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.3s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.3s finished


random_forest: scored 300000/377205 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 377205/377205 rows
random_forest: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/val_scored_rf_full.csv
(377205, 8)


In [18]:
def remove_gold_candidates_for_generation(scored_df):
    before = len(scored_df)
    filtered = scored_df[scored_df["source"] != "gold_distractor"].copy()
    after = len(filtered)

    print("Removed gold distractor candidates from generation pool:")
    print("Before:", before)
    print("After:", after)
    print("Removed:", before - after)

    return filtered

def candidate_similarity(a, b):
    return word_overlap(a, b)

def select_top3_for_group(group, diversity_lambda=0.25):
    group = group.copy()

    answer = safe_text(group["correct_answer"].iloc[0])
    group = group[
        group["candidate"].apply(lambda x: normalize_text(x) != normalize_text(answer))
    ]

    group = group.sort_values("score", ascending=False)

    selected = []

    for _, row in group.iterrows():
        cand = safe_text(row["candidate"])

        if not cand.strip():
            continue

        if normalize_text(cand) in [normalize_text(x["candidate"]) for x in selected]:
            continue

        too_similar = False

        for chosen in selected:
            sim = candidate_similarity(cand, chosen["candidate"])
            if sim >= diversity_lambda:
                too_similar = True
                break

        if too_similar:
            continue

        selected.append(row.to_dict())

        if len(selected) == 3:
            break

    if len(selected) < 3:
        for _, row in group.iterrows():
            cand = safe_text(row["candidate"])

            if normalize_text(cand) in [normalize_text(x["candidate"]) for x in selected]:
                continue

            selected.append(row.to_dict())

            if len(selected) == 3:
                break

    return selected

def select_top3_distractors(scored_df, output_path, model_name, diversity_lambda=0.25):
    if os.path.exists(output_path):
        os.remove(output_path)
        print("Deleted old top-3 file:", output_path)

    rows = []

    for qid, group in scored_df.groupby("question_uid"):
        selected = select_top3_for_group(group, diversity_lambda=diversity_lambda)

        base = group.iloc[0]

        out = {
            "question_uid": qid,
            "question": base["question"],
            "correct_answer": base["correct_answer"],
            "model": model_name
        }

        for i in range(3):
            if i < len(selected):
                out[f"generated_distractor_{i+1}"] = selected[i]["candidate"]
                out[f"generated_score_{i+1}"] = selected[i]["score"]
            else:
                out[f"generated_distractor_{i+1}"] = ""
                out[f"generated_score_{i+1}"] = np.nan

        rows.append(out)

    selected_df = pd.DataFrame(rows)
    selected_df.to_csv(output_path, index=False)

    print("Saved selected distractors:", output_path)
    print(selected_df.shape)

    return selected_df

print("Top-3 helpers ready.")

Top-3 helpers ready.


In [19]:
val_scored_lr_generation = remove_gold_candidates_for_generation(val_scored_lr)
val_scored_rf_generation = remove_gold_candidates_for_generation(val_scored_rf)

lr_val_selected_path = f"{MODEL_B_RESULTS_DIR}/val_top3_logreg_generation_only_full.csv"
rf_val_selected_path = f"{MODEL_B_RESULTS_DIR}/val_top3_rf_generation_only_full.csv"

val_top3_lr = select_top3_distractors(
    val_scored_lr_generation,
    lr_val_selected_path,
    "logistic_regression",
    diversity_lambda=0.25
)

val_top3_rf = select_top3_distractors(
    val_scored_rf_generation,
    rf_val_selected_path,
    "random_forest",
    diversity_lambda=0.25
)

Removed gold distractor candidates from generation pool:
Before: 377205
After: 350850
Removed: 26355
Removed gold distractor candidates from generation pool:
Before: 377205
After: 350850
Removed: 26355
Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/val_top3_logreg_generation_only_full.csv
(8785, 10)
Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/val_top3_rf_generation_only_full.csv
(8785, 10)


In [20]:
smooth = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def normalize_for_eval(text):
    text = normalize_text(remove_punctuation(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def get_reference_distractor_map(original_df):
    ref_map = {}

    for _, row in original_df.iterrows():
        refs = [
            safe_text(row["distractor_1"]),
            safe_text(row["distractor_2"]),
            safe_text(row["distractor_3"])
        ]
        refs = [r for r in refs if r.strip()]
        ref_map[row["question_uid"]] = refs

    return ref_map

def compute_text_generation_metrics(generated, reference):
    gen_tokens = tokenize_words(generated)
    ref_tokens = tokenize_words(reference)

    if not gen_tokens or not ref_tokens:
        return {
            "bleu1": 0.0,
            "bleu2": 0.0,
            "bleu4": 0.0,
            "rouge1_f1": 0.0,
            "rouge2_f1": 0.0,
            "rougeL_f1": 0.0,
            "meteor": 0.0
        }

    bleu1 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(1, 0, 0, 0),
        smoothing_function=smooth
    )

    bleu2 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smooth
    )

    bleu4 = sentence_bleu(
        [ref_tokens],
        gen_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smooth
    )

    rouge_scores = rouge.score(reference, generated)

    try:
        meteor = meteor_score([ref_tokens], gen_tokens)
    except:
        meteor = 0.0

    return {
        "bleu1": float(bleu1),
        "bleu2": float(bleu2),
        "bleu4": float(bleu4),
        "rouge1_f1": float(rouge_scores["rouge1"].fmeasure),
        "rouge2_f1": float(rouge_scores["rouge2"].fmeasure),
        "rougeL_f1": float(rouge_scores["rougeL"].fmeasure),
        "meteor": float(meteor)
    }

def best_match_metrics(generated, references):
    best_metrics = None
    best_score = -1

    for ref in references:
        metrics = compute_text_generation_metrics(generated, ref)
        score = metrics["meteor"] + metrics["rougeL_f1"] + metrics["bleu1"]

        if score > best_score:
            best_score = score
            best_metrics = metrics

    if best_metrics is None:
        return {
            "bleu1": 0.0,
            "bleu2": 0.0,
            "bleu4": 0.0,
            "rouge1_f1": 0.0,
            "rouge2_f1": 0.0,
            "rougeL_f1": 0.0,
            "meteor": 0.0
        }

    return best_metrics

def evaluate_generated_distractors_bleu_rouge_meteor(selected_df, original_df, model_name):
    ref_map = get_reference_distractor_map(original_df)

    metric_rows = []

    for _, row in selected_df.iterrows():
        qid = row["question_uid"]
        refs = ref_map.get(qid, [])

        generated_distractors = [
            safe_text(row["generated_distractor_1"]),
            safe_text(row["generated_distractor_2"]),
            safe_text(row["generated_distractor_3"])
        ]

        generated_distractors = [g for g in generated_distractors if g.strip()]

        for gen in generated_distractors:
            metrics = best_match_metrics(gen, refs)
            metrics["question_uid"] = qid
            metrics["generated_distractor"] = gen
            metric_rows.append(metrics)

    metrics_df = pd.DataFrame(metric_rows)

    summary = {
        "model": model_name,
        "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
        "questions_evaluated": int(selected_df["question_uid"].nunique()),
        "generated_distractors_evaluated": int(len(metrics_df)),
        "bleu1": float(metrics_df["bleu1"].mean()),
        "bleu2": float(metrics_df["bleu2"].mean()),
        "bleu4": float(metrics_df["bleu4"].mean()),
        "rouge1_f1": float(metrics_df["rouge1_f1"].mean()),
        "rouge2_f1": float(metrics_df["rouge2_f1"].mean()),
        "rougeL_f1": float(metrics_df["rougeL_f1"].mean()),
        "meteor": float(metrics_df["meteor"].mean())
    }

    return summary, metrics_df

print("Evaluation helpers ready.")

Evaluation helpers ready.


In [21]:
lr_val_generation_metrics, lr_val_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    val_top3_lr,
    val_df,
    "logistic_regression"
)

rf_val_generation_metrics, rf_val_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    val_top3_rf,
    val_df,
    "random_forest"
)

distractor_generation_val_comparison = pd.DataFrame([
    lr_val_generation_metrics,
    rf_val_generation_metrics
])

distractor_generation_val_comparison_path = f"{MODEL_B_RESULTS_DIR}/model_b_distractor_generation_validation_bleu_rouge_meteor_full.csv"

distractor_generation_val_comparison.to_csv(
    distractor_generation_val_comparison_path,
    index=False
)

display(distractor_generation_val_comparison)

best_row = distractor_generation_val_comparison.sort_values(
    by=["meteor", "rougeL_f1", "bleu1"],
    ascending=False
).iloc[0]

BEST_DISTRACTOR_MODEL_NAME = best_row["model"]

if BEST_DISTRACTOR_MODEL_NAME == "logistic_regression":
    best_distractor_model = logreg_ranker
else:
    best_distractor_model = rf_ranker

print("Best distractor model based on BLEU/ROUGE/METEOR:")
print(BEST_DISTRACTOR_MODEL_NAME)
print(best_row.to_dict())
print("Saved validation comparison:", distractor_generation_val_comparison_path)

,model,evaluation_type,questions_evaluated,generated_distractors_evaluated,bleu1,bleu2,bleu4,rouge1_f1,rouge2_f1,rougeL_f1,meteor
0,logistic_regression,BLEU_ROUGE_METEOR_for_generated_distractors,8785,26355,0.012734,0.006143,0.003723,0.04090,0.003365,0.040239,0.020605
1,random_forest,BLEU_ROUGE_METEOR_for_generated_distractors,8785,26355,0.014235,0.007210,0.004146,0.04536,0.005147,0.044679,0.023031


Best distractor model based on BLEU/ROUGE/METEOR:
random_forest
{'model': 'random_forest', 'evaluation_type': 'BLEU_ROUGE_METEOR_for_generated_distractors', 'questions_evaluated': 8785, 'generated_distractors_evaluated': 26355, 'bleu1': 0.01423451100961536, 'bleu2': 0.007210287080436371, 'bleu4': 0.004146463690893888, 'rouge1_f1': 0.0453600956440166, 'rouge2_f1': 0.005147139394519622, 'rougeL_f1': 0.04467884772097952, 'meteor': 0.02303074050324878}
Saved validation comparison: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_distractor_generation_validation_bleu_rouge_meteor_full.csv


In [22]:
best_test_scored_path = f"{MODEL_B_RESULTS_DIR}/test_scored_{BEST_DISTRACTOR_MODEL_NAME}_full.csv"

if os.path.exists(best_test_scored_path):
    os.remove(best_test_scored_path)
    print("Deleted old test scored file:", best_test_scored_path)

test_scored_best = score_feature_file(
    best_distractor_model,
    test_features_path,
    test_candidates_path,
    best_test_scored_path,
    BEST_DISTRACTOR_MODEL_NAME,
    SCORE_CHUNK_SIZE
)

random_forest: total feature rows: 377353
random_forest: existing scored rows: 0


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.2s finished


random_forest: scored 100000/377353 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.3s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.3s finished


random_forest: scored 200000/377353 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.5s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    2.2s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    2.3s finished


random_forest: scored 300000/377353 rows


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.2s
[Parallel(n_jobs=2)]: Done 196 tasks      | elapsed:    1.0s
[Parallel(n_jobs=2)]: Done 200 out of 200 | elapsed:    1.0s finished


random_forest: scored 377353/377353 rows
random_forest: saved scored candidates: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/test_scored_random_forest_full.csv
(377353, 8)


In [23]:
test_scored_best_generation = remove_gold_candidates_for_generation(test_scored_best)

best_test_top3_path = f"{MODEL_B_RESULTS_DIR}/test_top3_{BEST_DISTRACTOR_MODEL_NAME}_generation_only_full.csv"

test_top3_best = select_top3_distractors(
    test_scored_best_generation,
    best_test_top3_path,
    BEST_DISTRACTOR_MODEL_NAME,
    diversity_lambda=0.25
)

Removed gold distractor candidates from generation pool:
Before: 377353
After: 350995
Removed: 26358
Saved selected distractors: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/test_top3_random_forest_generation_only_full.csv
(8786, 10)


In [24]:
test_distractor_generation_metrics, test_distractor_generation_details = evaluate_generated_distractors_bleu_rouge_meteor(
    test_top3_best,
    test_df,
    BEST_DISTRACTOR_MODEL_NAME
)

test_distractor_generation_metrics_path = f"{MODEL_B_RESULTS_DIR}/model_b_best_distractor_test_bleu_rouge_meteor_full.json"
test_distractor_generation_details_path = f"{MODEL_B_RESULTS_DIR}/model_b_best_distractor_test_generation_details_full.csv"

with open(test_distractor_generation_metrics_path, "w") as f:
    json.dump(test_distractor_generation_metrics, f, indent=2)

test_distractor_generation_details.to_csv(
    test_distractor_generation_details_path,
    index=False
)

print("Best distractor test BLEU/ROUGE/METEOR metrics:")
print(json.dumps(test_distractor_generation_metrics, indent=2))

print("Saved summary:", test_distractor_generation_metrics_path)
print("Saved details:", test_distractor_generation_details_path)

Best distractor test BLEU/ROUGE/METEOR metrics:
{
  "model": "random_forest",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_distractors",
  "questions_evaluated": 8786,
  "generated_distractors_evaluated": 26358,
  "bleu1": 0.014671629653086023,
  "bleu2": 0.0075783267800398875,
  "bleu4": 0.004280892781640539,
  "rouge1_f1": 0.04590476061102177,
  "rouge2_f1": 0.0055073406873501875,
  "rougeL_f1": 0.045034412376216706,
  "meteor": 0.023799971014266317
}
Saved summary: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_best_distractor_test_bleu_rouge_meteor_full.json
Saved details: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_best_distractor_test_generation_details_full.csv


In [25]:
hint_generation_test_metrics_path = f"{MODEL_B_RESULTS_DIR}/model_b_hint_generation_test_bleu_rouge_meteor_full.json"
hint_generation_test_details_path = f"{MODEL_B_RESULTS_DIR}/model_b_hint_generation_test_details_full.csv"

if not os.path.exists(hint_generation_test_metrics_path):
    raise FileNotFoundError("Hint metrics file missing. Only then would hint evaluation need rerun.")

with open(hint_generation_test_metrics_path, "r") as f:
    hint_generation_test_metrics = json.load(f)

hint_generation_test_details = pd.read_csv(hint_generation_test_details_path)

print("Loaded hint generation metrics:")
print(json.dumps(hint_generation_test_metrics, indent=2))

print("\nHint details shape:", hint_generation_test_details.shape)
display(hint_generation_test_details.head(5))

Loaded hint generation metrics:
{
  "model": "Logistic Regression Hint Ranker",
  "evaluation_type": "BLEU_ROUGE_METEOR_for_generated_hints",
  "important_limitation": "RACE has no gold human-written hints, so the reference hint is a proxy: the most relevant passage sentence with the correct answer masked.",
  "questions_evaluated": 8786,
  "bleu1": 0.29553114699769656,
  "bleu2": 0.27734729376701284,
  "bleu4": 0.25695117753951857,
  "rouge1_f1": 0.43800344820527587,
  "rouge2_f1": 0.39865122367358485,
  "rougeL_f1": 0.4337181534059404,
  "meteor": 0.6869130950039368
}

Hint details shape: (8786, 14)


,bleu1,bleu2,bleu4,rouge1_f1,rouge2_f1,rougeL_f1,meteor,question_uid,question,correct_answer,reference_hint,generated_hint_1,generated_hint_2,generated_hint_3
0,0.322581,0.311086,0.285823,0.487805,0.461538,0.487805,0.826033,q_1378,"according to the passage, some people think th...",lose weight,you will [ANSWER] more [ANSWER] if you your ot...,Focus on the part of the passage related to: a...,A relevant sentence says: you will [ANSWER] mo...,The answer is supported by this context: you w...
1,0.210526,0.187317,0.126737,0.347826,0.285714,0.347826,0.721591,q_19784,the writer was surprised to find the woman's w...,clean,her window was [ANSWER]!,Focus on the part of the passage related to: w...,A relevant sentence says: her window was [ANSW...,The answer is supported by this context: her w...
2,0.372093,0.364541,0.348482,0.548387,0.533333,0.548387,0.855511,q_16858,"according to the passage, the fa(football asso...",each may.,the most interesting part of the english footb...,Focus on the part of the passage related to: a...,A relevant sentence says: the most interesting...,The answer is supported by this context: the m...
3,0.054054,0.012254,0.005960,0.078431,0.000000,0.078431,0.092025,q_31050,attention exchange was not a major concern in ...,natural checks and balances,"in traditional societies, with smaller populat...",Focus on the part of the passage related to: a...,A relevant sentence says: there were [ANSWER]:...,The answer is supported by this context: there...
4,0.403509,0.398147,0.386941,0.575000,0.564103,0.575000,0.871176,q_6565,"according to the passage, a happy person is mo...",a kind helper,"only when you are at peace with yourself, will...",Focus on the part of the passage related to: a...,A relevant sentence says: only when you are at...,The answer is supported by this context: only ...


In [26]:
examples_df = test_top3_best.merge(
    hint_generation_test_details[[
        "question_uid",
        "reference_hint",
        "generated_hint_1",
        "generated_hint_2",
        "generated_hint_3"
    ]],
    on="question_uid",
    how="left"
)

examples_df = examples_df.merge(
    test_df[[
        "question_uid",
        "article",
        "distractor_1",
        "distractor_2",
        "distractor_3"
    ]],
    on="question_uid",
    how="left"
)

examples_path = f"{MODEL_B_RESULTS_DIR}/model_b_qualitative_examples_full.csv"
examples_df.to_csv(examples_path, index=False)

display(examples_df[[
    "question",
    "correct_answer",
    "generated_distractor_1",
    "generated_distractor_2",
    "generated_distractor_3",
    "distractor_1",
    "distractor_2",
    "distractor_3",
    "generated_hint_1",
    "generated_hint_2",
    "generated_hint_3"
]].head(10))

print("Saved qualitative examples:", examples_path)

,question,correct_answer,generated_distractor_1,generated_distractor_2,generated_distractor_3,distractor_1,distractor_2,distractor_3,generated_hint_1,generated_hint_2,generated_hint_3
0,before the writer came to the high school summ...,student,introduced,attention boy,noticed,instructor,camper,reporter,Focus on the part of the passage related to: w...,A relevant sentence says: in the summer betwee...,The answer is supported by this context: in th...
1,where can you often find volunteers in the uni...,in a hospital.,high school college,orphanages,nation,at a bus stop.,in a park.,in a shop.,Focus on the part of the passage related to: v...,"A relevant sentence says: for example, some hi...",The answer is supported by this context: for e...
2,how do volunteers usually help those who are s...,"they mow their lawns, do their shopping and cl...",school college students,boys girls,orphanages,"they cook, sew or wash their clothes.",they tell them stories and sing and dance for ...,"they clean, wax(......)and repair their cars.",Focus on the part of the passage related to: v...,"A relevant sentence says: they paint, [ANSWER]...",The answer is supported by this context: they ...
3,the parrot was expensive because _ .,it could talk with the people,talk man asked,pulled string,thats,it was pretty beautiful,it was a very unusual bird,it was in the middle of the store,Focus on the part of the passage related to: p...,A relevant sentence says: once there was a par...,The answer is supported by this context: once ...
4,which of the following is not right?,mr zhao's has the best food and the most comfo...,seats comfortable,lis cools,zhaos people,mr li's has the worst noodles and the most com...,mr cool's has the hardest seats and the worst ...,these three restaurants are in a busy town.,Focus on the part of the passage related to: f...,A relevant sentence says: mrs zhao's has the [...,The answer is supported by this context: mrs z...
5,she likes to eat _ and bread in the morning.,"eggs, milk, bananas",salad,likes eat,pies,"eggs, chicken, milk","eggs, milk, apples","milk, pizza",Focus on the part of the passage related to: l...,A relevant sentence says: she has [ANSWER] and...,The answer is supported by this context: she h...
6,in england the traffic drives _ .,on the left,left morning,street look,floors,on the right,in the middle,in the front,Focus on the part of the passage related to: e...,A relevant sentence says: when you are in engl...,The answer is supported by this context: when ...
7,"which of the following should you say ""no"" whe...",stay alone at home as much as possible,yourselfi,anythingit,theres,you should always look for the good things of ...,learn something new and go for it,keep a diary to express your feelings,Focus on the part of the passage related to: f...,A relevant sentence says: when you are feeling...,The answer is supported by this context: when ...
8,"according to the passage, recycling language m...",using vocabulary that we have learnt very often,way recycle,favourite articles,sentences,repeating vocabulary at times,revising vocabulary at a proper time,learning new vocabulary as much as possible,Focus on the part of the passage related to: a...,A relevant sentence says: recycling language m...,The answer is supported by this context: recyc...
9,when is lu han's birthday?,on april 20th.,no47,playing soccer game,exo,on april 18th.,on april 19th.,on april 25th.,Focus on the part of the passage related to: h...,A relevant sentence says: lu han celebrated hi...,The answer is supported by this context: lu ha...


Saved qualitative examples: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_qualitative_examples_full.csv


In [27]:
model_b_summary = {
    "model_b_task": "Distractor and Hint Generator",
    "updated_evaluation_policy": "Final reporting uses BLEU, ROUGE, and METEOR only, following the latest project announcement.",

    "data_files": {
        "train": TRAIN_PATH,
        "val": VAL_PATH,
        "test": TEST_PATH
    },

    "full_mode": FULL_MODE,

    "distractor_generation": {
        "candidate_extraction": "Unigrams, bigrams, trigrams, and frequency-ranked content phrases extracted from the passage.",
        "positive_training_candidates": "Original RACE wrong options: distractor_1, distractor_2, distractor_3.",
        "negative_training_candidates": "Extracted non-answer passage candidates.",
        "important_evaluation_fix": "Gold/reference distractors are used for training labels but removed from validation/test generation candidate pools before top-3 selection.",
        "features": feature_cols,
        "models_trained": [
            "Logistic Regression",
            "Random Forest"
        ],
        "validation_generation_metrics": {
            "logistic_regression": lr_val_generation_metrics,
            "random_forest": rf_val_generation_metrics
        },
        "selected_model": BEST_DISTRACTOR_MODEL_NAME,
        "selection_criteria": "Highest validation METEOR, then ROUGE-L F1, then BLEU-1.",
        "test_generation_metrics": test_distractor_generation_metrics,
        "diversity_penalty": 0.25
    },

    "hint_generation": {
        "method": "ML-scored extractive sentence ranking with graduated hint templates.",
        "features": [
            "cos_question_sentence",
            "cos_answer_sentence",
            "cos_question_answer_sentence",
            "question_sentence_overlap",
            "answer_sentence_overlap",
            "sentence_position_norm",
            "sentence_word_len",
            "contains_correct_answer"
        ],
        "model": "Logistic Regression",
        "evaluation_limitation": "RACE does not provide gold human-written hints. Proxy reference hints are created from the most relevant passage sentence with the answer masked.",
        "test_generation_metrics": hint_generation_test_metrics
    },

    "artifacts": {
        "tfidf_vectorizer": tfidf_path,
        "logistic_regression_distractor_ranker": logreg_ranker_path,
        "random_forest_distractor_ranker": rf_ranker_path,
        "test_top3_outputs": best_test_top3_path,
        "distractor_test_metrics": test_distractor_generation_metrics_path,
        "hint_test_metrics": hint_generation_test_metrics_path,
        "qualitative_examples": examples_path
    }
}

summary_path = f"{MODEL_B_RESULTS_DIR}/model_b_final_summary_full.json"

with open(summary_path, "w") as f:
    json.dump(model_b_summary, f, indent=2)

print("Saved Model B final summary:", summary_path)
print(json.dumps(model_b_summary, indent=2))

Saved Model B final summary: /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_final_summary_full.json
{
  "model_b_task": "Distractor and Hint Generator",
  "updated_evaluation_policy": "Final reporting uses BLEU, ROUGE, and METEOR only, following the latest project announcement.",
  "data_files": {
    "train": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_train_full.csv",
    "val": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_val_full.csv",
    "test": "/content/drive/MyDrive/race_rc_project/data/processed/model_b_test_full.csv"
  },
  "full_mode": true,
  "distractor_generation": {
    "candidate_extraction": "Unigrams, bigrams, trigrams, and frequency-ranked content phrases extracted from the passage.",
    "positive_training_candidates": "Original RACE wrong options: distractor_1, distractor_2, distractor_3.",
    "negative_training_candidates": "Extracted non-answer passage candidates.",
    "important_evaluation_fix": "

In [28]:
final_files = [
    train_candidates_path,
    train_features_path,
    val_candidates_path,
    val_features_path,
    test_candidates_path,
    test_features_path,

    logreg_ranker_path,
    rf_ranker_path,

    lr_val_scored_path,
    rf_val_scored_path,
    lr_val_selected_path,
    rf_val_selected_path,

    distractor_generation_val_comparison_path,
    best_test_scored_path,
    best_test_top3_path,
    test_distractor_generation_metrics_path,
    test_distractor_generation_details_path,

    hint_generation_test_metrics_path,
    hint_generation_test_details_path,

    examples_path,
    summary_path
]

print("Final Model B files:")
for p in final_files:
    print(os.path.exists(p), "->", p)

Final Model B files:
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_train_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_train_features_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_val_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_val_features_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_test_candidates_full.csv
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/model_b_test_features_full.csv
True -> /content/drive/MyDrive/race_rc_project/models/model_b/logistic_regression_distractor_ranker_full.pkl
True -> /content/drive/MyDrive/race_rc_project/models/model_b/random_forest_distractor_ranker_full.pkl
True -> /content/drive/MyDrive/race_rc_project/results/model_b_true_full/val_scored_logreg_full.csv
True -> /content/drive/MyDrive/race_r